# Semana 9 — Pandas I: exploración y filtrado

**Programación para Ciencia de Datos**

Las semanas 6 y 7 trabajaron con NumPy: un `array` por columna, localizada por
posición, y todo del mismo tipo. La semana 8 trabajó con regex: un valor de
texto a la vez. Las dos cosas funcionan, y las dos te obligan a lo mismo —abrir
el archivo, partir cada línea, localizar la columna, convertir, y no perder de
vista qué fila era cada cosa.

**pandas** trae la pieza que faltaba: la **tabla completa**, con las columnas
localizadas por nombre, cada una con su propio tipo, y las filas alineadas entre
sí. El archivo que en la semana 6 te costó veinte líneas abrir se carga en una,
y la estadística de la semana 7 y los patrones de la semana 8 se aplican a una
columna entera de un jalón.

Esta semana son tres preguntas, y son las tres de siempre cuando llega un
dataset nuevo:

| Pregunta | Herramienta |
|---|---|
| ¿Qué hay aquí adentro? | `read_csv`, `shape`, `dtypes`, `info`, `describe`, `value_counts` |
| ¿Qué pedazo quiero ver? | `loc`, `iloc`, máscaras booleanas, `isin`, `between` |
| ¿Qué tan sucio está, exactamente? | el accessor `.str` con los patrones de la semana 8 |

> **De aquí sale la Práctica 4.** La P4 es pandas + regex: explorar tu dataset,
> reportar la calidad de cada columna y validar `email` y `telefono`. La tabla de
> diagnóstico que armamos el viernes es la mitad del entregable.

> **Lo que esta semana *no* cubre:** `fillna`, `dropna`, `astype`,
> `drop_duplicates`, `to_datetime`, `groupby`, `merge`, `pivot_table` y `apply`
> son de la **semana 10**. La diferencia entre las dos semanas es una sola
> palabra: esta semana **diagnosticas** —cuentas qué está mal y dónde—, la que
> entra **corriges**. Resistir la tentación de corregir hoy es parte del
> ejercicio: no se arregla lo que todavía no acabaste de medir.

**Mapa de la semana**

| Sesión | Tema | Lo que vas a poder hacer al final |
|---|---|---|
| 1 | `Series`, `DataFrame`, `read_csv` y la primera mirada | Cargar un CSV sucio y describir qué tiene |
| 2 | Máscaras booleanas, `isin`, `between`, `value_counts`, `sort_values` | Quedarte con las filas que cumplen una condición |
| 3 | Taller: el accessor `.str` y los patrones de la semana 8 | La tabla de diagnóstico de calidad que pide la P4 |

> **Cómo usar este notebook.** Los ejemplos van sobre un CSV de 24 filas que se
> escribe aquí abajo, no sobre tu dataset. Tiene **las mismas 9 columnas** que el
> tuyo y la misma clase de basura, pero cabe en la pantalla: cuando pandas te
> diga que hay 5 correos mal formados y 1 celda vacía puedes contarlos con el
> dedo y comprobar que no te está mintiendo. Correrlo sobre tus 103,000 filas es
> tu trabajo y va en los ejercicios.

### Los datos de hoy

`ventas_ejemplo.csv` es el archivo de la semana 6 con las **9 columnas** del
dataset real del curso. Los valores están inventados; los **problemas** son
exactamente los de tu CSV: celdas vacías, un texto metido en una columna
numérica, el mismo producto escrito de cuatro maneras, un monto imposible, un
minuto negativo, seis formatos de fecha, correos roídos, teléfonos escritos de
nueve maneras y una fila repetida completa.

Y está separado por `|`, como el tuyo, porque `direccion` y
`comentario_cliente` traen comas adentro.

In [ ]:
%%writefile ventas_ejemplo.csv
id|producto|monto_venta|minutos_en_sitio|timestamp|comentario_cliente|email|telefono|direccion
1001|Laptop|15000.50|45.0|2024-12-08 02:13:41|Excelente equipo, llego a tiempo|ana.flores78@outlook.com|80-4512-0612|Av. Juarez 120, Col. Centro, CDMX
1002|Mouse|350|12.0|2024-11-19 10:05:00|Bueno|luis_jimenez20@gmail.com|+52 43-7814-0090|Calle 5 de Mayo 8, Monterrey
1003|Laptop|14500||16/08/2023 02:06|Caro pero vale la pena|mariarodriguez86yahoo.com|(24) 7544-5493|Blvd. Reforma 45, Guadalajara
1004|Teclado|800|8.5|2024-03-02 18:44:10|Sin comentarios||15 3019 3755|Priv. Hidalgo 3, Puebla
1005|Monitor|5000|33.0|2024-07-21 09:12:30||sofia.perez@yahoo.com|5533672345|Calle Morelos 77, Toluca
1006|Laptop||51.0|20240118|Muy satisfecho con la compra|diego.ramirez11@hotmail|57.4604.7817|Av. Universidad 900, CDMX
1007|Mouse|400|5.0|2024-05-30 21:00:00|Cumple|val.ortiz@outlook.com|96-8256-8|Calle Allende 14, Queretaro
1008|Impresora|3500|27.5|2024-02-14 13:30:15|Tarda en imprimir, pero sirve|andrea.luna@gmail.com|55-2456-2345|Av. Patria 210, Zapopan
1009|Mouse|320||05-23-2024|Regular|miguel.cruz@gmail.com||Calle Zaragoza 56, Leon
1010|Laptop|16000|62.0|2024-09-04 08:01:59|La recomiendo|paula.rodriguez17@|52-8915-9923|Av. Chapultepec 18, CDMX
1011|Teclado|750|9.0|2024-10-11 16:22:44|Buena relacion precio valor|jose.gonzalez33@live.com|89-6943-0962|Calle Guerrero 101, Merida
1012|Monitor|no disponible|40.0|2024-06-18 11:11:11|Imagen nitida|renatareyes62@@outlook.com|(55) 1234 5678|Av. Insurgentes 1500, CDMX
1013|laptop|15200|48.0|28-09-24 03:15:05|Llego golpeado el empaque|carlos.vega@gmail.com|81-3344-5566|Calle Colon 9, Veracruz
1014|LAPTOP|14800|53.0|2024-04-25 07:45:00|Rapida y silenciosa|fer.mendoza@gmail.com|33 2211 4455|Av. Vallarta 300, Guadalajara
1015| Laptop |15100|47.0|2024-08-09 19:30:21|Cumple lo que promete|nadia.soto@outlook.com|444-123-4567|Calle Pino 22, San Luis Potosi
1016|Impresora|999999|31.0|2024-01-07 06:00:00|Precio raro en el sitio|hugo.barrios@gmail.com|228-765-4321|Av. Xalapa 55, Xalapa
1017|Mouse|-250|7.0|2024-11-02 22:15:40|Devolucion registrada|lucia.pena@hotmail.com|55 9876 5432|Calle Sol 4, CDMX
1018|Teclado|820|-3.0|2024-10-28 14:08:12|Teclas duras|erick.rios@gmail.com|998-555-1212|Av. Tulum 80, Cancun
1019|Monitor|5400|35.0|28 de febrero de 2024|Buen tamano|juan.hernandez65 @ live.com|477-321-9876|Calle Hierro 61, Leon
1020|Mouse|380|6.5|2024-12-01 12:00:00|Nada que reportar|karla.diaz@yahoo.com|55-7788-9900|Av. Revolucion 2, CDMX
1021|Laptop|15500|49.0|2024-03-19 17:55:05|Excelente servicio|omar.castillo@gmail.com|222-444-6666|Calle Puebla 33, Puebla
1021|Laptop|15500|49.0|2024-03-19 17:55:05|Excelente servicio|omar.castillo@gmail.com|222-444-6666|Calle Puebla 33, Puebla
1022|Impresora|3600|29.0||Buena para oficina|silvia.nava@outlook.com|33-1010-2020|Av. Lopez Mateos 700, Zapopan
1023|Monitor|4900|32.5|2024-07-07 07:07:07|Se calienta un poco|raul.mejia@gmail.com|81-2020-3030|Calle Acero 18, Monterrey

In [1]:
import pandas as pd
import numpy as np
import re

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.3
numpy : 2.4.2


> **La versión de pandas importa esta semana.** Las salidas de este notebook son
> de **pandas 3.x**, donde las columnas de texto tienen el tipo `str`. Si tu
> `pd.__version__` empieza con `2.`, esas columnas te van a aparecer como
> `object` y hay dos lugares donde el comportamiento cambia de verdad; los dos
> están marcados abajo con una nota. Todo lo demás es idéntico. Si quieres
> igualar lo que ves aquí: `pip install -U pandas`.

## Sesión 1 — `Series`, `DataFrame` y la primera mirada

Dos estructuras, y con ellas se trabaja todo el resto del semestre:

- una **`Series`** es una columna: un array con etiquetas y nombre
- un **`DataFrame`** es la tabla: varias `Series` que comparten las etiquetas de
  las filas

### Una `Series` es un array con etiquetas

In [2]:
s = pd.Series([15000.50, 350, 14500, 800], name="monto_venta")
print(s)
print()
print("valores:", s.values)
print("indice :", list(s.index))
print("dtype  :", s.dtype)
print("nombre :", s.name)

0    15000.5
1      350.0
2    14500.0
3      800.0
Name: monto_venta, dtype: float64

valores: [15000.5   350.  14500.    800. ]
indice : [0, 1, 2, 3]
dtype  : float64
nombre : monto_venta


Sin decirle nada, pandas le puso el índice `0, 1, 2, 3` —la posición, igual que
un array. La diferencia es que ese índice **se puede cambiar**, y entonces los
valores se buscan por etiqueta:

In [ ]:
s2 = pd.Series([15000.50, 350, 14500, 800],
               index=["laptop", "mouse", "monitor", "teclado"],
               name="monto_venta")
print(s2)
print()
print('por etiqueta  s2.loc["mouse"]:', s2.loc["mouse"])
print("por posicion  s2.iloc[1] :", s2.iloc[1])
#print(s2["mouse"])  ##NO HACER ESTO


laptop     15000.5
mouse        350.0
monitor    14500.0
teclado      800.0
Name: monto_venta, dtype: float64

por etiqueta  s2.loc["mouse"]: 350.0
por posicion  s2.iloc[1] : 350.0
350.0


### Lo que una `Series` tiene y un array no

| | `np.array` | `pd.Series` |
|---|---|---|
| Acceso | solo por posición | por posición **y** por etiqueta |
| Tipo | uno para todo el array | uno por `Series`, y hay tipo de texto |
| Faltantes | hay que inventarles un `nan` | `NaN` de fábrica, y `isna()` para contarlos |
| Nombre | no tiene | `.name`, que se vuelve el nombre de la columna |
| Al operar dos | por posición, y truena si no miden igual | **se alinean por etiqueta** |

El último renglón es el que cambia cómo se programa. Dos `Series` no se suman
posición contra posición: se suman **etiqueta contra etiqueta**.

In [3]:
enero = pd.Series([100, 200, 300], index=["laptop", "mouse", "teclado"])
febrero = pd.Series([50, 500], index=["mouse", "monitor"])
print(enero, "\n")
print(febrero, "\n")
print(enero + febrero)

laptop     100
mouse      200
teclado    300
dtype: int64 

mouse       50
monitor    500
dtype: int64 

laptop       NaN
monitor      NaN
mouse      250.0
teclado      NaN
dtype: float64


Fíjate en lo que pasó: `mouse` sumó 250 —sus 200 de enero más sus 50 de
febrero— porque estaba en las dos, y las otras tres salieron `NaN` porque
faltaban en una. Con arrays de NumPy esto habría sido un error de tamaños, o
peor, una suma de peras con manzanas alineadas por casualidad.

> **Esa alineación es la razón de ser de pandas.** Cuando filtras, ordenas o
> agregas una columna, las filas no se te desacomodan: la etiqueta las amarra.
> Es también de donde salen los `NaN` que no esperabas, y lo primero que hay que
> revisar cuando una operación entre dos columnas devuelve puros faltantes.

### `read_csv`: la línea que reemplaza media semana 6

In [6]:
df = pd.read_csv("ventas_ejemplo.csv", sep="|")
df.head(10)

,id,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion
0,1001,Laptop,15000.50,45.0,2024-12-08 02:13:41,"Excelente equipo, llego a tiempo",ana.flores78@outlook.com,80-4512-0612,"Av. Juarez 120, Col. Centro, CDMX"
1,1002,Mouse,350,12.0,2024-11-19 10:05:00,Bueno,luis_jimenez20@gmail.com,+52 43-7814-0090,"Calle 5 de Mayo 8, Monterrey"
2,1003,Laptop,14500,NaN,16/08/2023 02:06,Caro pero vale la pena,mariarodriguez86yahoo.com,(24) 7544-5493,"Blvd. Reforma 45, Guadalajara"
3,1004,Teclado,800,8.5,2024-03-02 18:44:10,Sin comentarios,NaN,15 3019 3755,"Priv. Hidalgo 3, Puebla"
4,1005,Monitor,5000,33.0,2024-07-21 09:12:30,NaN,sofia.perez@yahoo.com,5533672345,"Calle Morelos 77, Toluca"
5,1006,Laptop,NaN,51.0,20240118,Muy satisfecho con la compra,diego.ramirez11@hotmail,57.4604.7817,"Av. Universidad 900, CDMX"
6,1007,Mouse,400,5.0,2024-05-30 21:00:00,Cumple,val.ortiz@outlook.com,96-8256-8,"Calle Allende 14, Queretaro"
7,1008,Impresora,3500,27.5,2024-02-14 13:30:15,"Tarda en imprimir, pero sirve",andrea.luna@gmail.com,55-2456-2345,"Av. Patria 210, Zapopan"
8,1009,Mouse,320,NaN,05-23-2024,Regular,miguel.cruz@gmail.com,NaN,"Calle Zaragoza 56, Leon"
9,1010,Laptop,16000,62.0,2024-09-04 08:01:59,La recomiendo,paula.rodriguez17@,52-8915-9923,"Av. Chapultepec 18, CDMX"


Eso es todo. Compáralo con la semana 6: `open`, `read`, `strip`, `split("\n")`,
`split("|")`, `cabecera.index("monto_venta")` y una lista por comprensión por
cada columna. pandas hizo las nueve de un golpe, le puso nombre a cada una y le
adivinó el tipo.

> **`sep="|"` no es opcional.** El separador por omisión de `read_csv` es la
> coma, y tu dataset no usa comas para separar: las usa *adentro* de `direccion`
> y del comentario. Si lo olvidas, pandas no se queja —te da una tabla de **una
> sola columna** cuyo nombre es toda la cabecera pegada:

In [4]:
mal = pd.read_csv("ventas_ejemplo.csv")   # sin sep: se lo come todo como una columna
print("forma:", mal.shape)
print("la unica columna se llama:")
print(" ", mal.columns[0])

forma: (24, 1)
la unica columna se llama:
  id|producto|monto_venta|minutos_en_sitio|timestamp|comentario_cliente|email|telefono|direccion


### La primera mirada: cuatro preguntas, cuatro llamadas

Cada vez que cargues un dataset nuevo, en este orden:

| Pregunta | Llamada |
|---|---|
| ¿De qué tamaño es? | `df.shape` |
| ¿Cómo se llaman las columnas? | `df.columns` |
| ¿De qué tipo quedó cada una? | `df.dtypes` |
| ¿Cuántos huecos tiene cada una? | `df.info()` |

In [7]:
print("forma (filas, columnas):", df.shape)
print("filas:", len(df))
print()
print("columnas:", df.columns.tolist())

forma (filas, columnas): (24, 9)
filas: 24

columnas: ['id', 'producto', 'monto_venta', 'minutos_en_sitio', 'timestamp', 'comentario_cliente', 'email', 'telefono', 'direccion']


In [8]:
print(df.dtypes)

id                      int64
producto                  str
monto_venta               str
minutos_en_sitio      float64
timestamp                 str
comentario_cliente        str
email                     str
telefono                  str
direccion                 str
dtype: object


### `monto_venta` quedó como texto, y no es un error de pandas

`minutos_en_sitio` quedó `float64` y `monto_venta` quedó texto. Las dos traen
números. La diferencia es una sola celda, la del `id` 1012:

```
1012|Monitor|no disponible|40.0|...
```

pandas le adivina el tipo a cada columna leyendo **todos** sus valores, con la
regla más conservadora que hay: si uno solo no es número, la columna entera es
texto. Es exactamente lo que te pasó en la semana 6 cuando `float(valor)` tronó
con una celda rara, solo que ahora lo descubres en la primera mirada y no diez
líneas después.

> **Lo mismo va a pasar con tu dataset**, y es el primer hallazgo que vale la
> pena anotar: *una de mis dos columnas numéricas no cargó como numérica, y
> estas son las celdas culpables.*

> **Si tu pandas es 2.x** ahí dirá `object` en lugar de `str`. `object` quiere
> decir "cada celda es un objeto de Python, quién sabe de qué tipo", y para todo
> lo de esta semana se comporta igual: es una columna de texto.

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  24 non-null     int64  
 1   producto            24 non-null     str    
 2   monto_venta         23 non-null     str    
 3   minutos_en_sitio    22 non-null     float64
 4   timestamp           23 non-null     str    
 5   comentario_cliente  23 non-null     str    
 6   email               23 non-null     str    
 7   telefono            23 non-null     str    
 8   direccion           24 non-null     str    
dtypes: float64(1), int64(1), str(7)
memory usage: 1.8 KB


`info()` junta todo en una sola salida, y la columna que de verdad importa es
**`Non-Null Count`**: 24 filas en total, pero `monto_venta` tiene 23,
`minutos_en_sitio` 22, `timestamp` 23, `comentario_cliente` 23, `email` 23,
`telefono` 23. Esa resta —24 menos lo que diga— es el número de celdas vacías de
cada columna, y es el primer renglón de tu reporte de calidad.

In [13]:
print("--- las primeras 3 ---")
print(df.head()[["id", "producto", "monto_venta", "minutos_en_sitio", "timestamp"]])
print()
print("--- las ultimas 2 ---")
print(df.tail(4)[["id", "producto", "monto_venta", "minutos_en_sitio"]])

--- las primeras 3 ---
     id producto monto_venta  minutos_en_sitio            timestamp
0  1001   Laptop    15000.50              45.0  2024-12-08 02:13:41
1  1002    Mouse         350              12.0  2024-11-19 10:05:00
2  1003   Laptop       14500               NaN     16/08/2023 02:06
3  1004  Teclado         800               8.5  2024-03-02 18:44:10
4  1005  Monitor        5000              33.0  2024-07-21 09:12:30

--- las ultimas 2 ---
      id   producto monto_venta  minutos_en_sitio
20  1021     Laptop       15500              49.0
21  1021     Laptop       15500              49.0
22  1022  Impresora        3600              29.0
23  1023    Monitor        4900              32.5


### `describe()`: la semana 7 en una línea

In [14]:
df.describe()

,id,minutos_en_sitio
count,24.00000,22.000000
mean,1012.37500,30.772727
std,6.88295,19.163288
min,1001.00000,-3.000000
25%,1006.75000,9.750000
50%,1012.50000,32.750000
75%,1018.25000,47.750000
max,1023.00000,62.000000


Ahí está toda la estadística que calculaste a mano en la semana 7 —`count`,
`mean`, `std`, los tres cuartiles, mínimo y máximo— para todas las columnas
numéricas de un golpe. Y la salida trae tres cosas que hay que saber leer:

1. **`monto_venta` no aparece.** `describe()` solo mira las columnas numéricas, y
   `monto_venta` es texto. Que una columna se ausente de aquí es una alarma, no
   un detalle.
2. **`id` sí aparece, y no significa nada.** El promedio de un identificador es
   un número perfectamente calculado y perfectamente inútil. pandas no sabe qué
   significan tus columnas; tú sí.
3. **`minutos_en_sitio` tiene mínimo `-3`.** Un minuto negativo no existe. Ese es
   un hallazgo real, y lo encontraste en la segunda llamada del día.

Para las columnas de texto hay que pedirlo aparte, y las medidas son otras:

In [ ]:
df[["producto", "email", "timestamp"]].describe()

`unique` es cuántos valores distintos hay, y `top`/`freq` el más repetido con su
número de apariciones. Dos cosas saltan: `producto` dice **8** valores distintos
cuando el catálogo tiene 5, y `email` tiene uno que aparece **2** veces cuando
debería ser único. Las dos se persiguen en la sesión 2.

### Seleccionar columnas: `Series` contra `DataFrame`

In [17]:
una = df["producto"]                      # un corchete    -> Series
dos = df[["producto", "monto_venta"]]     # doble corchete -> DataFrame

print("un corchete    ->", type(una).__name__, "| forma:", una.shape)
print("doble corchete ->", type(dos).__name__, "| forma:", dos.shape)

un corchete    -> Series | forma: (24,)
doble corchete -> DataFrame | forma: (24, 2)


> **La trampa del doble corchete.** `df["producto"]` es una columna: una
> `Series`, con `.str`, `.mean()` y `.value_counts()`. `df[["producto"]]` es una
> tabla de una columna: un `DataFrame`, sin `.str` y sin `.value_counts()`. El
> error se ve así: `AttributeError: 'DataFrame' object has no attribute 'str'`.
> Cuando lo veas, cuenta los corchetes.
>
> Y los dos corchetes están pegados porque adentro va una **lista** de nombres:
> el de afuera es la selección, el de adentro es la lista.

### `loc` y `iloc`: etiqueta contra posición

Dos formas de pedir filas, y la confusión entre ellas es el error más común del
semestre:

- **`df.loc[...]`** busca por **etiqueta** del índice
- **`df.iloc[...]`** busca por **posición**, como una lista de Python

Mientras el índice sea `0, 1, 2, ...` las dos dan lo mismo y parece que da igual
cuál uses. Deja de dar igual en cuanto filtras.

In [22]:
df.head(6)

,id,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion
0,1001,Laptop,15000.50,45.0,2024-12-08 02:13:41,"Excelente equipo, llego a tiempo",ana.flores78@outlook.com,80-4512-0612,"Av. Juarez 120, Col. Centro, CDMX"
1,1002,Mouse,350,12.0,2024-11-19 10:05:00,Bueno,luis_jimenez20@gmail.com,+52 43-7814-0090,"Calle 5 de Mayo 8, Monterrey"
2,1003,Laptop,14500,NaN,16/08/2023 02:06,Caro pero vale la pena,mariarodriguez86yahoo.com,(24) 7544-5493,"Blvd. Reforma 45, Guadalajara"
3,1004,Teclado,800,8.5,2024-03-02 18:44:10,Sin comentarios,NaN,15 3019 3755,"Priv. Hidalgo 3, Puebla"
4,1005,Monitor,5000,33.0,2024-07-21 09:12:30,NaN,sofia.perez@yahoo.com,5533672345,"Calle Morelos 77, Toluca"
5,1006,Laptop,NaN,51.0,20240118,Muy satisfecho con la compra,diego.ramirez11@hotmail,57.4604.7817,"Av. Universidad 900, CDMX"


In [21]:
print('df.loc[0, "monto_venta"] :', df.loc[0, "monto_venta"])
print("df.iloc[0, 2]         :", df.iloc[0, 2])    # fila 0, columna 1
print()
print("--- una fila, solo algunas columnas ---")
print(df.loc[5, ["id", "producto", "monto_venta", "email"]])

df.loc[0, "monto_venta"] : 15000.50
df.iloc[0, 2]         : 15000.50

--- una fila, solo algunas columnas ---
id                                1006
producto                        Laptop
monto_venta                        NaN
email          diego.ramirez11@hotmail
Name: 5, dtype: object


In [23]:
df.head()

,id,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion
0,1001,Laptop,15000.50,45.0,2024-12-08 02:13:41,"Excelente equipo, llego a tiempo",ana.flores78@outlook.com,80-4512-0612,"Av. Juarez 120, Col. Centro, CDMX"
1,1002,Mouse,350,12.0,2024-11-19 10:05:00,Bueno,luis_jimenez20@gmail.com,+52 43-7814-0090,"Calle 5 de Mayo 8, Monterrey"
2,1003,Laptop,14500,NaN,16/08/2023 02:06,Caro pero vale la pena,mariarodriguez86yahoo.com,(24) 7544-5493,"Blvd. Reforma 45, Guadalajara"
3,1004,Teclado,800,8.5,2024-03-02 18:44:10,Sin comentarios,NaN,15 3019 3755,"Priv. Hidalgo 3, Puebla"
4,1005,Monitor,5000,33.0,2024-07-21 09:12:30,NaN,sofia.perez@yahoo.com,5533672345,"Calle Morelos 77, Toluca"


In [24]:
print("--- loc: el final SI se incluye ---")
print(df.loc[0:2, ["id", "producto"]])
print()
print("--- iloc: el final NO se incluye, como en las listas ---")
print(df.iloc[0:2, 0:2])

--- loc: el final SI se incluye ---
     id producto
0  1001   Laptop
1  1002    Mouse
2  1003   Laptop

--- iloc: el final NO se incluye, como en las listas ---
     id producto
0  1001   Laptop
1  1002    Mouse


> **`loc` incluye el final del rango y `iloc` no.** `df.loc[0:2]` da tres filas;
> `df.iloc[0:2]` da dos. No es un descuido de pandas: un rango de etiquetas se
> lee "de la 0 a la 2", y un rango de posiciones se lee como un `slice` de
> Python. Hay que acordarse de las dos.

Y ahora la razón de verdad para no confundirlas: **filtrar no renumera el
índice.**

In [27]:
df["monto"] = pd.to_numeric(df["monto_venta"], errors="coerce")
df.head(15)

,id,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion,monto
0,1001,Laptop,15000.50,45.0,2024-12-08 02:13:41,"Excelente equipo, llego a tiempo",ana.flores78@outlook.com,80-4512-0612,"Av. Juarez 120, Col. Centro, CDMX",15000.5
1,1002,Mouse,350,12.0,2024-11-19 10:05:00,Bueno,luis_jimenez20@gmail.com,+52 43-7814-0090,"Calle 5 de Mayo 8, Monterrey",350.0
2,1003,Laptop,14500,NaN,16/08/2023 02:06,Caro pero vale la pena,mariarodriguez86yahoo.com,(24) 7544-5493,"Blvd. Reforma 45, Guadalajara",14500.0
3,1004,Teclado,800,8.5,2024-03-02 18:44:10,Sin comentarios,NaN,15 3019 3755,"Priv. Hidalgo 3, Puebla",800.0
4,1005,Monitor,5000,33.0,2024-07-21 09:12:30,NaN,sofia.perez@yahoo.com,5533672345,"Calle Morelos 77, Toluca",5000.0
5,1006,Laptop,NaN,51.0,20240118,Muy satisfecho con la compra,diego.ramirez11@hotmail,57.4604.7817,"Av. Universidad 900, CDMX",NaN
6,1007,Mouse,400,5.0,2024-05-30 21:00:00,Cumple,val.ortiz@outlook.com,96-8256-8,"Calle Allende 14, Queretaro",400.0
7,1008,Impresora,3500,27.5,2024-02-14 13:30:15,"Tarda en imprimir, pero sirve",andrea.luna@gmail.com,55-2456-2345,"Av. Patria 210, Zapopan",3500.0
8,1009,Mouse,320,NaN,05-23-2024,Regular,miguel.cruz@gmail.com,NaN,"Calle Zaragoza 56, Leon",320.0
9,1010,Laptop,16000,62.0,2024-09-04 08:01:59,La recomiendo,paula.rodriguez17@,52-8915-9923,"Av. Chapultepec 18, CDMX",16000.0


In [28]:
df.describe()

,id,minutos_en_sitio,monto
count,24.00000,22.000000,22.000000
mean,1012.37500,30.772727,52162.250000
std,6.88295,19.163288,211805.604331
min,1001.00000,-3.000000,-250.000000
25%,1006.75000,9.750000,762.500000
50%,1012.50000,32.750000,4950.000000
75%,1018.25000,47.750000,15075.125000
max,1023.00000,62.000000,999999.000000


In [29]:

baratos = df[df["monto"] < 1000]

print("indice de las filas que quedaron:", baratos.index.tolist())
print()
print("baratos.iloc[0] -> la primera que quedo, id:", baratos.iloc[0]["id"])
try:
    baratos.loc[0]
except KeyError:
    print("baratos.loc[0]  -> KeyError: la etiqueta 0 ya no esta en este subconjunto")

indice de las filas que quedaron: [1, 3, 6, 8, 10, 16, 17, 19]

baratos.iloc[0] -> la primera que quedo, id: 1002
baratos.loc[0]  -> KeyError: la etiqueta 0 ya no esta en este subconjunto


In [30]:
baratos

,id,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion,monto
1,1002,Mouse,350,12.0,2024-11-19 10:05:00,Bueno,luis_jimenez20@gmail.com,+52 43-7814-0090,"Calle 5 de Mayo 8, Monterrey",350.0
3,1004,Teclado,800,8.5,2024-03-02 18:44:10,Sin comentarios,NaN,15 3019 3755,"Priv. Hidalgo 3, Puebla",800.0
6,1007,Mouse,400,5.0,2024-05-30 21:00:00,Cumple,val.ortiz@outlook.com,96-8256-8,"Calle Allende 14, Queretaro",400.0
8,1009,Mouse,320,NaN,05-23-2024,Regular,miguel.cruz@gmail.com,NaN,"Calle Zaragoza 56, Leon",320.0
10,1011,Teclado,750,9.0,2024-10-11 16:22:44,Buena relacion precio valor,jose.gonzalez33@live.com,89-6943-0962,"Calle Guerrero 101, Merida",750.0
16,1017,Mouse,-250,7.0,2024-11-02 22:15:40,Devolucion registrada,lucia.pena@hotmail.com,55 9876 5432,"Calle Sol 4, CDMX",-250.0
17,1018,Teclado,820,-3.0,2024-10-28 14:08:12,Teclas duras,erick.rios@gmail.com,998-555-1212,"Av. Tulum 80, Cancun",820.0
19,1020,Mouse,380,6.5,2024-12-01 12:00:00,Nada que reportar,karla.diaz@yahoo.com,55-7788-9900,"Av. Revolucion 2, CDMX",380.0


In [31]:
tardados = df[df["minutos_en_sitio"] > 50]
tardados

,id,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion,monto
5,1006,Laptop,NaN,51.0,20240118,Muy satisfecho con la compra,diego.ramirez11@hotmail,57.4604.7817,"Av. Universidad 900, CDMX",NaN
9,1010,Laptop,16000,62.0,2024-09-04 08:01:59,La recomiendo,paula.rodriguez17@,52-8915-9923,"Av. Chapultepec 18, CDMX",16000.0
13,1014,LAPTOP,14800,53.0,2024-04-25 07:45:00,Rapida y silenciosa,fer.mendoza@gmail.com,33 2211 4455,"Av. Vallarta 300, Guadalajara",14800.0


Las filas se llevaron su etiqueta original: el subconjunto empieza en la
etiqueta **1**, no en la 0. Entonces `baratos.iloc[0]` es la primera fila que
quedó, y `baratos.loc[0]` ya no existe.

La regla, sin excepciones:

| Quiero | Uso |
|---|---|
| la fila **que trae esta etiqueta** | `df.loc[etiqueta]` |
| la fila **que va en este lugar** | `df.iloc[posicion]` |
| la primera / la última de lo que me quedó | `df.iloc[0]` / `df.iloc[-1]` |

En la práctica: para seleccionar por condición y para asignar, `loc`. Para "dame
la primera" y para recorrer por posición, `iloc`.

### El índice: `set_index`

Si tu tabla tiene una columna identificadora, puedes volverla el índice y buscar
por ella directamente:

In [32]:
df.head()

,id,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion,monto
0,1001,Laptop,15000.50,45.0,2024-12-08 02:13:41,"Excelente equipo, llego a tiempo",ana.flores78@outlook.com,80-4512-0612,"Av. Juarez 120, Col. Centro, CDMX",15000.5
1,1002,Mouse,350,12.0,2024-11-19 10:05:00,Bueno,luis_jimenez20@gmail.com,+52 43-7814-0090,"Calle 5 de Mayo 8, Monterrey",350.0
2,1003,Laptop,14500,NaN,16/08/2023 02:06,Caro pero vale la pena,mariarodriguez86yahoo.com,(24) 7544-5493,"Blvd. Reforma 45, Guadalajara",14500.0
3,1004,Teclado,800,8.5,2024-03-02 18:44:10,Sin comentarios,NaN,15 3019 3755,"Priv. Hidalgo 3, Puebla",800.0
4,1005,Monitor,5000,33.0,2024-07-21 09:12:30,NaN,sofia.perez@yahoo.com,5533672345,"Calle Morelos 77, Toluca",5000.0


In [33]:
por_id = df.set_index("id")

print("nombre del indice:", por_id.index.name)
print()
print(por_id.loc[1005, ["producto", "monto_venta", "email"]])

nombre del indice: id

producto                     Monitor
monto_venta                     5000
email          sofia.perez@yahoo.com
Name: 1005, dtype: str


In [35]:
por_id

,producto,monto_venta,minutos_en_sitio,timestamp,comentario_cliente,email,telefono,direccion,monto
id,,,,,,,,,
1001,Laptop,15000.50,45.0,2024-12-08 02:13:41,"Excelente equipo, llego a tiempo",ana.flores78@outlook.com,80-4512-0612,"Av. Juarez 120, Col. Centro, CDMX",15000.5
1002,Mouse,350,12.0,2024-11-19 10:05:00,Bueno,luis_jimenez20@gmail.com,+52 43-7814-0090,"Calle 5 de Mayo 8, Monterrey",350.0
1003,Laptop,14500,NaN,16/08/2023 02:06,Caro pero vale la pena,mariarodriguez86yahoo.com,(24) 7544-5493,"Blvd. Reforma 45, Guadalajara",14500.0
1004,Teclado,800,8.5,2024-03-02 18:44:10,Sin comentarios,NaN,15 3019 3755,"Priv. Hidalgo 3, Puebla",800.0
1005,Monitor,5000,33.0,2024-07-21 09:12:30,NaN,sofia.perez@yahoo.com,5533672345,"Calle Morelos 77, Toluca",5000.0
1006,Laptop,NaN,51.0,20240118,Muy satisfecho con la compra,diego.ramirez11@hotmail,57.4604.7817,"Av. Universidad 900, CDMX",NaN
1007,Mouse,400,5.0,2024-05-30 21:00:00,Cumple,val.ortiz@outlook.com,96-8256-8,"Calle Allende 14, Queretaro",400.0
1008,Impresora,3500,27.5,2024-02-14 13:30:15,"Tarda en imprimir, pero sirve",andrea.luna@gmail.com,55-2456-2345,"Av. Patria 210, Zapopan",3500.0
1009,Mouse,320,NaN,05-23-2024,Regular,miguel.cruz@gmail.com,NaN,"Calle Zaragoza 56, Leon",320.0


Conviene cuando vas a consultar filas por su identificador, y **solo si ese
identificador es único**. El de este archivo no lo es —el `1021` aparece dos
veces— y pandas te deja hacerlo igual: `por_id.loc[1021]` devuelve **dos** filas
en lugar de una, y cualquier cuenta que hagas encima queda inflada.

In [36]:
print(por_id.loc[1021, ["producto", "monto_venta"]])
print()
print("tipo de lo que devolvio:", type(por_id.loc[1021]).__name__)
print("tipo para un id unico  :", type(por_id.loc[1005]).__name__)

     producto monto_venta
id                       
1021   Laptop       15500
1021   Laptop       15500

tipo de lo que devolvio: DataFrame
tipo para un id unico  : Series


> **`set_index` no es obligatorio** y esta semana casi no hace falta: el índice
> `0..n-1` está bien para todo. Lo que sí tienes que saber es que **existe**, que
> `loc` lo usa, y que de ahí salen tanto el `KeyError` de arriba como esta
> sorpresa: la **misma llamada** devuelve una `Series` o un `DataFrame`
> dependiendo de si el id estaba repetido.

### Ejercicios sesión 1

Escribe las respuestas en las celdas. Los ejercicios que dicen *tu dataset* se
corren sobre `datos/{tema}-ruido.csv` de tu repositorio, con las ~103,000 filas
completas.

In [ ]:
# Ejercicio 1: Carga tu datos/{tema}-ruido.csv con read_csv y sep="|" e imprime
# shape, columns.tolist() y dtypes. Contesta por escrito, en un comentario:
#   - cuantas filas y columnas tiene
#   - cual de tus dos columnas numericas NO cargo como numerica, si alguna

# Tu codigo aqui:

In [ ]:
# Ejercicio 2: Corre info() sobre tu dataset y arma la lista de celdas vacias por
# columna (tu numero de filas menos el Non-Null Count de cada una). Di cual es la
# columna mas incompleta y con cuantos huecos.

# Tu codigo aqui:

In [ ]:
# Ejercicio 3: Corre describe() sobre tu dataset y busca tres cosas:
#   - un minimo o un maximo imposible (negativo, o absurdamente grande)
#   - una columna numerica que NO aparecio en la salida, y por que no aparecio
#   - una columna que si aparecio pero cuyo promedio no significa nada

# Tu codigo aqui:

In [ ]:
# Ejercicio 4: Predice que devuelve cada linea ANTES de correrla, y luego
# correla sobre df (el de 24 filas). Donde falles, escribe por que:
#   type(df["producto"])
#   type(df[["producto"]])
#   df.loc[0:3].shape
#   df.iloc[0:3].shape
#   df.loc[0, "email"] == df.iloc[0, 6]

# Tu codigo aqui:

In [ ]:
# Ejercicio 5: De tu dataset, imprime la primera y la ultima fila de las columnas
# <categorica>, <numerica_1> y email usando iloc. Luego pon id como indice con
# set_index y consulta una fila por su id. Si la consulta te devuelve un
# DataFrame en lugar de una Series, ese id esta duplicado: reportalo.

# Tu codigo aqui:

## Sesión 2 — Filtrar, ordenar y contar

La semana 7 cerró con máscaras booleanas sobre arrays de NumPy. Esta sesión es
la misma idea, con dos diferencias que la hacen mucho más útil: la máscara se
construye nombrando la columna, y al aplicarla te llevas **la fila completa** —
no solo los valores de la columna que preguntaste.

### Una comparación produce una `Series` de booleanos

In [37]:
mascara = df["minutos_en_sitio"] > 40

print(mascara)


0      True
1     False
2     False
3     False
4     False
5      True
6     False
7     False
8     False
9      True
10    False
11    False
12     True
13     True
14     True
15    False
16    False
17    False
18    False
19    False
20     True
21     True
22    False
23    False
Name: minutos_en_sitio, dtype: bool


In [38]:
print()
print("tipo    :", type(mascara).__name__, "| dtype:", mascara.dtype)
print("cuantas :", mascara.sum(), "filas cumplen")
print("fraccion:", round(mascara.mean(), 3))


tipo    : Series | dtype: bool
cuantas : 8 filas cumplen
fraccion: 0.333


Igual que en NumPy: `sum()` sobre booleanos cuenta los `True` y `mean()` da la
proporción. Y la máscara se usa para quedarte con las filas:

In [39]:
print(df.loc[mascara, ["id", "producto", "minutos_en_sitio"]])

      id  producto  minutos_en_sitio
0   1001    Laptop              45.0
5   1006    Laptop              51.0
9   1010    Laptop              62.0
12  1013    laptop              48.0
13  1014    LAPTOP              53.0
14  1015   Laptop               47.0
20  1021    Laptop              49.0
21  1021    Laptop              49.0


`df[mascara]` y `df.loc[mascara]` hacen lo mismo. Prefiere **`df.loc[mascara,
columnas]`**: es la única de las dos que te deja escoger columnas en la misma
línea, y es la forma que necesitas cuando además quieras **asignar** (al final de
la sesión).

### Primero convertir, luego comparar

`monto_venta` cargó como texto. Comparar texto con un número no se queda a
medias: truena.

In [40]:
"hola mundo" > 1000

TypeError: '>' not supported between instances of 'str' and 'int'

In [41]:
try:
    df["monto_venta"] > 1000
except TypeError as e:
    print("TypeError:", e)

TypeError: '>' not supported between instances of 'str' and 'int'


Ese `TypeError` es la buena noticia. La mala es lo que pasa si en lugar del
número escribes el número **entre comillas**, que es justo el dedazo que se
comete cuando uno ya sabe que la columna es texto:

In [42]:
alfabetico = df["monto_venta"] > "1000"     # texto contra texto: no se queja

print("dice que hay", alfabetico.sum(), "montos mayores a 1000")
print()
print(df.loc[alfabetico, "monto_venta"].tolist())

dice que hay 22 montos mayores a 1000

['15000.50', '350', '14500', '800', '5000', '400', '3500', '320', '16000', '750', 'no disponible', '15200', '14800', '15100', '999999', '820', '5400', '380', '15500', '15500', '3600', '4900']


Veintidós de veinticuatro, incluyendo `350`, `800`, `320` y el `no disponible`.
pandas no comparó números: comparó **cadenas, carácter por carácter**, y `"3"`
viene después de `"1"` en el alfabeto, así que `"350" > "1000"`. La máscara se
ve perfectamente bien y está perfectamente mal, y nada en la salida te lo dice.

> **Por eso la regla es convertir antes de comparar**, no corregir la
> comparación. En cuanto una columna numérica cargó como texto, lo primero que se
> hace con ella es `to_numeric`:

In [43]:
monto = pd.to_numeric(df["monto_venta"], errors="coerce")

print("dtype despues de convertir:", monto.dtype)
print("valores que no se pudieron convertir:", monto.isna().sum())
print()
print("las filas culpables:")
print(df.loc[monto.isna(), ["id", "monto_venta"]])

dtype despues de convertir: float64
valores que no se pudieron convertir: 2

las filas culpables:
      id    monto_venta
5   1006            NaN
11  1012  no disponible


In [44]:
monto.isna()

0     False
1     False
2     False
3     False
4     False
5      True
6     False
7     False
8     False
9     False
10    False
11     True
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
Name: monto_venta, dtype: bool

`errors="coerce"` significa *lo que no se pueda convertir, ponlo como `NaN`*.
La otra opción es `errors="raise"`, que es el comportamiento por omisión: truena
y te dice cuál fue el valor culpable. Útil cuando quieres enterarte; inservible
cuando son 103,000 filas y lo que necesitas es el conteo.

Las dos filas que salieron `NaN` son distintas entre sí, y la diferencia
importa:

| `id` | valor | qué es |
|---|---|---|
| 1006 | *(vacío)* | **faltante**: nunca hubo dato |
| 1012 | `no disponible` | **mal formado**: hay dato, pero no es un número |

Para `to_numeric` las dos son `NaN`. Para tu reporte de calidad son dos cosas
distintas y se cuentan aparte, exactamente como el *inválido* contra el *mal
escrito pero recuperable* de la semana 8.

> **Esta columna convertida es diagnóstico, no limpieza.** La guardamos en una
> variable para poder preguntarle cosas hoy. Decidir qué se hace con esos `NaN`
> —tirar la fila, rellenar, dejarla fuera del promedio— es la semana 10.

In [45]:
# La guardamos como columna nueva para tenerla a mano el resto del notebook
df["monto"] = monto

print(df["monto"].describe())

count        22.000000
mean      52162.250000
std      211805.604331
min        -250.000000
25%         762.500000
50%        4950.000000
75%       15075.125000
max      999999.000000
Name: monto, dtype: float64


Ahora sí: `count` 22 de 24 filas, un mínimo de **-250** y un máximo de
**999,999**. Un monto negativo y un monto de un millón en un catálogo donde lo
más caro es una laptop de 16,000. Los dos son candidatos a outlier con la regla
de las k desviaciones de la semana 7 —`std` arriba de 200,000 es la pista—, y
los dos son hallazgos para el reporte.

### Combinar condiciones: `&`, `|`, `~`

In [46]:
caras = (df["monto"] > 10000) & (df["minutos_en_sitio"] > 45)
print("caras y con mucho tiempo en sitio:", caras.sum())
print(df.loc[caras, ["id", "producto", "monto", "minutos_en_sitio"]])

caras y con mucho tiempo en sitio: 6
      id  producto    monto  minutos_en_sitio
9   1010    Laptop  16000.0              62.0
12  1013    laptop  15200.0              48.0
13  1014    LAPTOP  14800.0              53.0
14  1015   Laptop   15100.0              47.0
20  1021    Laptop  15500.0              49.0
21  1021    Laptop  15500.0              49.0


In [ ]:
raras = (df["monto"] < 0) | (df["monto"] > 100000) | (df["minutos_en_sitio"] < 0)
print("filas con algun valor imposible:", raras.sum())
print(df.loc[raras, ["id", "producto", "monto", "minutos_en_sitio"]])

> **Los paréntesis no son opcionales.** `&` y `|` aprietan más que `>` y `<`, así
> que sin paréntesis Python intenta evaluar `10000 & df["minutos_en_sitio"]`
> antes de la comparación y truena con un `TypeError` que no se parece en nada a
> lo que está mal. Cada condición, en su propio paréntesis.

Y la otra: los operadores son `&`, `|`, `~` — **no** `and`, `or`, `not`.

In [ ]:
try:
    (df["monto"] > 10000) and (df["minutos_en_sitio"] > 45)
except ValueError as e:
    print("ValueError:", str(e)[:70], "...")

`and` quiere un `True` o un `False` para decidir, y le estás dando 24 valores a
la vez: no sabe cuál usar. `&` sí sabe —compara elemento contra elemento— y por
eso es el que va.

| Quiero | Operador |
|---|---|
| las dos condiciones | `&` |
| alguna de las dos | `\|` |
| lo contrario de la máscara | `~` |

`~` es especialmente útil para "las que **no** cumplen":

In [ ]:
print("filas sin ningun valor imposible:", (~raras).sum())

### `isin`, `between`, `isna`

Tres atajos que evitan cadenas largas de `|`:

In [47]:
# isin: pertenece a esta lista (en vez de == o == o ==)
perifericos = df["producto"].isin(["Mouse", "Teclado"])
print("perifericos:", perifericos.sum())

# between: dentro del rango, con los extremos incluidos
medio = df["minutos_en_sitio"].between(10, 50)
print("entre 10 y 50 minutos:", medio.sum())

# isna / notna: el hueco
print("sin minutos registrados:", df["minutos_en_sitio"].isna().sum())
print("con minutos registrados:", df["minutos_en_sitio"].notna().sum())

perifericos: 8
entre 10 y 50 minutos: 13
sin minutos registrados: 2
con minutos registrados: 22


> **`between` incluye los dos extremos** (`inclusive="both"` es el valor por
> omisión). Si necesitas que no, se dice: `inclusive="neither"`, `"left"` o
> `"right"`.

> **Un `NaN` nunca cumple una comparación**, ni siquiera `NaN != NaN`. Por eso
> `(df["minutos_en_sitio"] > 40).sum()` cuenta 8 y las 2 filas sin dato no se
> cuelan como `True`. Lo que sí hacen es aparecer en `~mascara` —las 16
> restantes— mezcladas con las que de verdad están en 40 o menos, como si
> fueran lo mismo. "No cumple la condición" y "no tiene dato" son dos cosas
> distintas, y el `~` las junta. Para los faltantes, `isna()`, siempre aparte.

### Contar: `value_counts`, `nunique`, `unique`

`value_counts()` es la llamada que más vas a usar en tu vida con pandas:

In [ ]:
print(df["producto"].value_counts())

Ahí está el hallazgo que `describe()` había insinuado con su `unique: 8`. El
catálogo tiene **cinco** productos, pero hay **ocho** valores distintos porque
`Laptop` viene escrito de cuatro maneras:

| valor | veces | qué tiene |
|---|---|---|
| `Laptop` | 6 | bien |
| `laptop` | 1 | todo en minúscula |
| `LAPTOP` | 1 | todo en mayúsculas |
| `" Laptop "` | 1 | con espacios alrededor |

Las cuatro son el **mismo producto**. Cualquier conteo por categoría que hagas
hoy sobre esta columna reporta 6 laptops cuando son 9, y eso es un error del
33%. Lo arreglamos en la sesión 3 (ahí se ve cómo) y se aplica de verdad en la
semana 10.

> **Esto es lo primero que hay que correr sobre tu columna `<categorica>`.** Si
> `value_counts()` te devuelve más renglones que categorías de verdad tiene tu
> tema, acabas de encontrar trabajo.

In [ ]:
# dropna=False: los faltantes como un renglon mas, en vez de desaparecidos
print(df["email"].value_counts(dropna=False).head(4))
print()
print("--- cuantos valores distintos, sin listarlos ---")
for col in ["producto", "email", "telefono", "timestamp"]:
    print("%-12s %2d distintos, %d vacios" % (col, df[col].nunique(), df[col].isna().sum()))

> **`value_counts()` esconde los faltantes por omisión.** Con `dropna=False` los
> cuenta como un renglón más, etiquetado `NaN`. Para un reporte de calidad,
> siempre `dropna=False`: lo que estás reportando es precisamente lo que falta.
>
> `nunique()` también los ignora, y eso sí está bien: un hueco no es un valor
> distinto.

Y en proporción, con `normalize=True`:

In [ ]:
print((df["producto"].value_counts(normalize=True) * 100).round(1))

### Ordenar: `sort_values` y `nlargest`

In [ ]:
print(df.sort_values("monto", ascending=False)[["id", "producto", "monto"]].head(5))

In [ ]:
# nlargest: lo mismo, mas corto, cuando solo quieres los primeros
print(df.nlargest(3, "monto")[["id", "producto", "monto"]])
print()
print(df.nsmallest(3, "monto")[["id", "producto", "monto"]])

> **Los `NaN` se van al final**, suba o baje el orden. Si quieres verlos
> primero: `na_position="first"`. Y `sort_values` **no** modifica `df`: devuelve
> una tabla nueva. Si la quieres, guárdala en una variable.

Ordenar por dos columnas: una lista de nombres y una lista de direcciones.

In [ ]:
print(df.sort_values(["producto", "monto"], ascending=[True, False])
        .head(6)[["producto", "monto", "id"]])

### Columnas nuevas

Asignar una columna que no existe la crea. Las operaciones son vectorizadas,
igual que en NumPy:

In [ ]:
df["monto_por_minuto"] = (df["monto"] / df["minutos_en_sitio"]).round(2)

print(df[["id", "monto", "minutos_en_sitio", "monto_por_minuto"]].head(5))

Fíjate en la fila del `id` 1003: `minutos_en_sitio` está vacío, así que la
división dio `NaN`. No tronó, no avisó —se propagó. Es el comportamiento
correcto (no se puede dividir entre un dato que no tienes) y es la razón de
revisar `isna().sum()` **después** de cada columna calculada, no solo al cargar.

In [ ]:
print("filas sin monto_por_minuto:", df["monto_por_minuto"].isna().sum())
print("de ellas, por falta de monto  :", (df["monto"].isna()).sum())
print("de ellas, por falta de minutos:", (df["minutos_en_sitio"].isna()).sum())

### Las dos trampas de asignar sobre un filtro

Esta es la parte de la sesión que te va a ahorrar una tarde de confusión.
Filtrar devuelve una tabla **nueva**. Modificarla no modifica el original:

In [ ]:
mouses = df[df["producto"] == "Mouse"]
mouses["monto"] = 0            # modifica el subconjunto, no df

print("en el subconjunto:", mouses["monto"].tolist())
print("en df (intacto)  :", df.loc[df["producto"] == "Mouse", "monto"].tolist())

Eso casi siempre es lo que querías. La trampa es la versión en una sola línea,
que **no hace nada en absoluto**:

In [ ]:
df[df["producto"] == "Mouse"]["monto"] = 0     # no hace nada

print("df despues del intento:", df.loc[df["producto"] == "Mouse", "monto"].tolist())

Eso es la **asignación encadenada**: `df[mascara]` fabrica una tabla nueva,
le asignas la columna a esa tabla nueva, y la tabla nueva se descarta al
terminar la línea. pandas 3 te avisa con un `ChainedAssignmentError` (es una
advertencia, no detiene nada) y en pandas 2 el aviso era el famoso
`SettingWithCopyWarning`.

La forma correcta es `loc`, con la máscara y la columna **en la misma llamada**:

In [ ]:
df.loc[df["producto"] == "Mouse", "monto"] = 0

print("ahora si:", df.loc[df["producto"] == "Mouse", "monto"].tolist())

# Y lo dejamos como estaba, que el 0 era solo para la demostracion
df["monto"] = pd.to_numeric(df["monto_venta"], errors="coerce")
print("restaurado:", df.loc[df["producto"] == "Mouse", "monto"].tolist())

Las dos reglas, que son las mismas de siempre dichas al revés:

| Quiero | Escribo |
|---|---|
| **modificar** `df` en las filas que cumplen | `df.loc[mascara, "col"] = valor` |
| **un subconjunto aparte** para trabajarlo | `sub = df[mascara].copy()` |

> **El `.copy()` es por claridad, no por necesidad.** Desde pandas 3 el
> subconjunto ya es independiente del original de todos modos. Escribirlo deja
> dicho en el código que sabes que es una copia y que la querías.

### Ejercicios sesión 2

In [ ]:
# Ejercicio 6: Sobre tu dataset, convierte <numerica_1> con to_numeric(errors=
# "coerce") y reporta, por separado:
#   - cuantas celdas estaban vacias (isna sobre la columna ORIGINAL)
#   - cuantas tenian algo que no era un numero (NaN despues de convertir,
#     menos las que ya estaban vacias)
# Los dos numeros van en tu reporte de calidad, y no son el mismo numero.

# Tu codigo aqui:

In [ ]:
# Ejercicio 7: Con una sola mascara combinada, cuenta las filas de tu dataset que
# tienen ALGUN valor imposible: <numerica_1> negativa, <numerica_2> negativa, o
# cualquiera de las dos por arriba de un umbral que tu decidas y justifiques.
# Imprime las primeras 10 con loc y las columnas que lo demuestran.

# Tu codigo aqui:

In [ ]:
# Ejercicio 8: Corre value_counts(dropna=False) sobre tu columna <categorica> y
# contesta:
#   - cuantos valores distintos reporta
#   - cuantas categorias de verdad tiene tu tema
#   - cuales renglones son el mismo valor escrito distinto
# Si los dos primeros numeros coinciden, dilo: tambien es un resultado.

# Tu codigo aqui:

In [ ]:
# Ejercicio 9: Predice el resultado de cada linea ANTES de correrla, sobre el df
# de 24 filas. Tres de las cinco tronan: di cual es el error antes de verlo.
#   (df["monto"] > 1000).sum()
#   (df["monto_venta"] > 1000).sum()
#   df[df["producto"] == "Laptop" & df["monto"] > 15000]
#   df["minutos_en_sitio"].between(0, 60).sum()
#   df["producto"].isin("Mouse").sum()

# Tu codigo aqui:

In [ ]:
# Ejercicio 10: Arma, sobre tu dataset, la tabla de los 5 registros con
# <numerica_1> mas alta y los 5 con la mas baja (nlargest y nsmallest), con las
# columnas id, <categorica> y <numerica_1>. Mirandolas, di cuales te parecen
# datos reales y cuales errores de captura, y por que.

# Tu codigo aqui:

## Sesión 3 — Taller: el diagnóstico de calidad, columna por columna

La semana 8 cerró con esta tabla y la promesa de que el patrón no cambia, solo
cambia quién lo aplica. Hoy se cobra la promesa:

| Semana 8, valor por valor | Hoy, columna completa |
|---|---|
| `re.fullmatch(RE_EMAIL, v) is not None` | `df["email"].str.fullmatch(RE_EMAIL)` |
| `re.search(p, v)` | `df["texto"].str.contains(p)` |
| `m.groupdict()` | `df["timestamp"].str.extract(p)` |
| `re.sub(r"\D", "", v)` | `df["telefono"].str.replace(r"\D", "", regex=True)` |

La columna de la derecha no tiene bucles, no tiene `if`, y devuelve una `Series`
alineada con `df`: se puede usar como máscara en el mismo renglón en que la
calculas.

El entregable de la sesión es una **tabla de diagnóstico**: por cada columna,
cuántos valores hay, cuántos faltan y cuántos están mal formados. Eso es la
mitad del entregable de la P4.

### El accessor `.str`

Las `Series` de texto tienen un accessor `.str` que expone los métodos de las
cadenas de Python, aplicados a toda la columna:

In [ ]:
print("--- longitud de cada direccion ---")
print(df["direccion"].str.len().head(4))
print()
print("--- a minusculas ---")
print(df["producto"].str.lower().head(4).tolist())
print()
print("--- sin espacios alrededor ---")
print(repr(df["producto"].iloc[14]), "->", repr(df["producto"].str.strip().iloc[14]))

> **`.str` solo existe en `Series` de texto.** Si lo pides sobre una columna
> numérica, `AttributeError: Can only use .str accessor with string values`. Y si
> lo pides sobre `df[["producto"]]` —doble corchete— tampoco, porque eso es un
> `DataFrame`.

**Los faltantes se quedan faltantes.** Un `NaN` no es una cadena, así que `.str`
no lo toca: el resultado para esa fila es `NaN` otra vez.

In [ ]:
print("el email de la fila 3 es:", repr(df["email"].iloc[3]))
print(".str.len() de esa fila   :", df["email"].str.len().iloc[3])
print(".str.upper() de esa fila :", repr(df["email"].str.upper().iloc[3]))
print()
print("dtype de .str.len():", df["email"].str.len().dtype, "<- float, por el NaN")

Ese `float64` donde esperabas enteros es la huella de un faltante: una columna de
longitudes no puede ser entera si alguna fila no tiene texto que medir. Es la
misma historia de la semana 6 con `np.nan` forzando `float`.

### `producto`: el mismo valor escrito de cuatro maneras

Con dos llamadas en cadena se ve el tamaño real del problema que encontró
`value_counts()` en la sesión 2:

In [ ]:
normalizado = df["producto"].str.strip().str.capitalize()

print("--- como viene ---")
print(df["producto"].value_counts().to_string())
print()
print("--- normalizado ---")
print(normalizado.value_counts().to_string())

De ocho valores distintos a cinco, y las laptops pasaron de 6 a **9**. Ese 33%
es el error que habrías reportado si hubieras contado por categoría sin mirar
primero.

> **Hoy esto se mide, no se guarda.** Fíjate en que `normalizado` es una variable
> aparte: `df["producto"]` sigue sucio. Reemplazar la columna es *limpieza*, y la
> limpieza es la semana 10 —junto con la decisión de si `Capitalize` es la forma
> correcta o si hay productos que se escriben con dos palabras y se van a romper.

### `email`: el patrón de la semana 8, ahora sobre la columna

Este es literalmente el `RE_EMAIL` que armaste el viernes pasado, copiado tal
cual:

In [ ]:
RE_EMAIL = (
    r"[A-Za-z0-9._%+-]+"        # usuario
    r"@"                        # la arroba, exactamente una
    r"[A-Za-z0-9-]+"            # dominio
    r"(?:\.[A-Za-z0-9-]+)*"     # subdominios opcionales: .com.mx
    r"\.[A-Za-z]{2,}"           # extension: punto y al menos dos letras
)

valido = df["email"].str.fullmatch(RE_EMAIL)

print("dtype de la mascara:", valido.dtype)
print(valido.value_counts().to_string())

> **Si tu pandas es 2.x**, aquí hay una diferencia real. En 2.x la máscara sale
> de tipo `object` con un `NaN` en la fila vacía, y usarla para filtrar truena
> con `Cannot mask with non-boolean array containing NA / NaN values`. La
> solución es pasarle `na=False`: `.str.fullmatch(RE_EMAIL, na=False)`.
> Escribirlo no estorba en pandas 3, así que si en tu equipo hay las dos
> versiones, pónganlo siempre.

**Y los vacíos cuentan como `False`.** Esos 6 que no pasaron incluyen la celda
vacía de la fila 3, que no está *mal formada*: no está. Si reportas "6 correos
mal formados" estás juntando dos problemas distintos en un número, y es
precisamente la distinción que la P4 te pide separar.

La cuenta correcta son **tres** números que suman el total:

In [ ]:
hay = df["email"].notna()

total = len(df)
vacios = (~hay).sum()
validos = (valido & hay).sum()
mal_formados = (~valido & hay).sum()

print("total        :", total)
print("vacios       :", vacios)
print("validos      :", validos)
print("mal formados :", mal_formados)
print("suman        :", vacios + validos + mal_formados, "<- tiene que dar el total")

In [ ]:
# Y a mirarlos: 6 de 24 cabe en la pantalla, y hay que ver que el patron
# rechaza lo que debe rechazar y no otra cosa
print(df.loc[~valido & hay, ["id", "email"]].to_string(index=False))

Los cinco son errores de verdad —sin arroba, dos arrobas, sin dominio, sin
extensión, con espacios alrededor de la arroba— y son los mismos cinco casos del
catálogo de la semana 8. El patrón no se tuvo que tocar para pasar de una lista
de 8 valores a una columna de un dataset.

> **Mirar los rechazados no es opcional.** Un validador que rechaza el 40% de tu
> columna casi siempre es un patrón mal escrito, no un dataset 40% roto. La
> única forma de saber cuál de las dos es leer los rechazados.

### `telefono`: limpiar primero, validar después

La otra lección de la semana 8: cuando el valor es **recuperable**, no se valida
el formato original —se normaliza y se valida el resultado. Con `.str` son dos
llamadas:

In [ ]:
digitos = df["telefono"].str.replace(r"\D", "", regex=True)

print(digitos.head(8).to_string())

> **`regex=True` es obligatorio.** Sin él, `.str.replace` busca `\D`
> **literalmente** —una barra invertida seguida de una D— y no encuentra nada. No
> avisa: te devuelve la columna igualita, y tú sigues adelante creyendo que la
> limpiaste. Es la versión en pandas de la `r"..."` olvidada.

Ya con solo dígitos, validar es contar:

In [ ]:
largo = digitos.str.len()

print(largo.value_counts(dropna=False).sort_index().to_string())

Eso se lee de un golpe: **21** teléfonos de 10 dígitos (el formato nacional),
**1** de 12 (los 10 con el `52` de país), **1** de 7 (incompleto, irrecuperable)
y **1** vacío. Clasificarlos es una máscara por caso:

In [ ]:
estado = np.select(
    [
        df["telefono"].isna(),
        largo == 10,
        (largo == 12) & digitos.str.startswith("52"),
    ],
    ["vacio", "valido", "valido con lada"],
    default="mal formado",
)
df["telefono_estado"] = estado

print(df["telefono_estado"].value_counts().to_string())
print()
print(df.loc[df["telefono_estado"] == "mal formado", ["id", "telefono"]].to_string(index=False))

`np.select` toma una lista de condiciones y una de resultados, y se queda con el
**primer** `True` de cada fila —como un `if/elif/else` vectorizado. El orden
importa: `isna()` va primero porque si no, un vacío caería en `default` y se
reportaría como mal formado.

Y ahora la comparación que justifica todo el rodeo de limpiar antes de validar:

In [ ]:
oficial = df["telefono"].str.fullmatch(r"\d{2}-\d{4}-\d{4}").sum()
recuperables = (df["telefono_estado"] != "mal formado") & df["telefono"].notna()

print("pasan el formato 'oficial' tal como vienen:", oficial)
print("recuperables despues de limpiar          :", recuperables.sum())
print("buenos que habrias tirado                :", recuperables.sum() - oficial)
print()
print("las nueve formas en que viene escrita la columna:")
print(df["telefono"].dropna().str.replace(r"\d", "d", regex=True).unique().tolist())

> **De 24 teléfonos escritos de nueve maneras distintas, 22 son recuperables.**
> Si los hubieras validado directamente contra el formato "oficial"
> `\d{2}-\d{4}-\d{4}`, solo **8** habrían pasado: 14 teléfonos perfectamente
> buenos a la basura, un 58% de la columna. Esa es la diferencia entre
> *inválido* y *mal escrito pero recuperable*, y es la decisión que va
> documentada en tu README.

### `timestamp`: `.str.extract` con grupos con nombre

`extract` devuelve un **`DataFrame`**: una columna por grupo de captura, con el
nombre del grupo como nombre de la columna.

In [ ]:
partes = df["timestamp"].str.extract(r"(?P<anio>\d{4})-(?P<mes>\d{2})-(?P<dia>\d{2})")

print(type(partes).__name__, partes.shape)
print(partes.head(7).to_string())

Las filas que no cumplen el patrón salen `NaN` en todas sus columnas —no tronan,
no se saltan— y ahí está la cuenta que importa:

In [ ]:
print("valores que NO estan en formato ISO:", partes["anio"].isna().sum())
print()
print(df.loc[partes["anio"].isna(), ["id", "timestamp"]].to_string(index=False))

Seis: cuatro formatos alternativos, uno en palabras y el vacío. Para contar
cuántos hay de cada formato, el catálogo de la semana 8 con `.str.contains`:

In [ ]:
FORMATOS = {
    "iso           (2024-12-08 02:13:41)": r"^\d{4}-\d{2}-\d{2}",
    "compacto      (20240118)":            r"^\d{8}$",
    "diagonales    (16/08/2023 02:06)":    r"^\d{2}/\d{2}/\d{4}",
    "mes primero   (05-23-2024)":          r"^\d{2}-\d{2}-\d{4}",
    "anio corto    (28-09-24 03:15:05)":   r"^\d{2}-\d{2}-\d{2}\b",
    "en palabras   (28 de febrero de...)": r"de\s+[a-z]+\s+de",
}

encontrados = 0
for nombre, patron in FORMATOS.items():
    n = df["timestamp"].str.contains(patron, regex=True).sum()
    encontrados += n
    print("%-38s %2d" % (nombre, n))

print("%-38s %2d" % ("vacios", df["timestamp"].isna().sum()))
print("%-38s %2d" % ("suma", encontrados + df["timestamp"].isna().sum()))

La suma da 24, el total de filas: **cada valor cayó en exactamente un formato**.
Y eso no pasa por suerte: pasa porque cada patrón empieza con `^`. Quítales el
ancla y mira lo que sale:

In [ ]:
suma = 0
for nombre, patron in FORMATOS.items():
    sin_ancla = patron.lstrip("^")
    n = df["timestamp"].str.contains(sin_ancla, regex=True).sum()
    suma += n
    print("%-38s %2d" % (nombre.split("(")[0].strip(), n))

print("%-38s %2d  <- y solo hay 24 filas" % ("suma", suma + df["timestamp"].isna().sum()))

**42 de 24 filas.** El culpable es el patrón de año corto: sin el ancla,
`\d{2}-\d{2}-\d{2}\b` encuentra `24-12-08` **adentro** de `2024-12-08 02:13:41`,
así que los 18 valores en ISO se contaron dos veces.

> **Por eso la verificación es la suma.** Si los conteos de tu catálogo de
> formatos suman más que tus filas, tienes patrones que se pisan y el reporte
> está inflado. Si suman menos, hay valores que ningún patrón reconoció y te
> falta un renglón en el catálogo. Solo cuando da exacto puedes decir que
> clasificaste la columna.

> **Convertir las fechas de verdad es `pd.to_datetime`, y es semana 10.** Hoy
> solo cuentas en qué formato vienen. Y de una vez ve el problema que te espera:
> `05-23-2024` es mes primero y `16/08/2023` es día primero. ¿Cuál es
> `10-09-2024`? Ningún patrón lo resuelve. Se decide, se documenta y se cuenta,
> como en la semana 8.

### Duplicados

`duplicated()` marca las filas repetidas. Por omisión marca la **segunda** y
siguientes apariciones, no la primera:

In [ ]:
print("filas identicas en las 9 columnas:", df.duplicated(subset=df.columns[:9]).sum())
print("ids repetidos                    :", df["id"].duplicated().sum())
print()
# keep=False marca TODAS las apariciones, que es lo que quieres para verlas
repetidos = df["id"].duplicated(keep=False)
print(df.loc[repetidos, ["id", "producto", "monto_venta", "email"]].to_string(index=False))

Un `id` que se repite en un dataset donde el `id` es la llave es un duplicado de
captura, y cada cuenta que hagas lo cuenta dos veces. En este archivo son dos
filas idénticas en las nueve columnas; en el tuyo, con 103,000 filas, puede
haber filas con el mismo `id` y valores **distintos**, que es un problema peor
porque hay que decidir cuál se queda.

> **Hoy se cuentan, no se borran.** `drop_duplicates` es semana 10, y la decisión
> de qué hacer cuando las dos filas no son idénticas también.

### La tabla de diagnóstico

Junta todo lo de la sesión en una sola tabla: una fila por columna, con lo que
tiene y lo que le falta. Esto es lo que va en tu reporte de la P4.

In [ ]:
diagnostico = pd.DataFrame({
    "dtype": df[df.columns[:9]].dtypes.astype(str),
    "no_vacios": df[df.columns[:9]].notna().sum(),
    "vacios": df[df.columns[:9]].isna().sum(),
    "distintos": df[df.columns[:9]].nunique(),
})
diagnostico["pct_vacios"] = (diagnostico["vacios"] / len(df) * 100).round(1)

print(diagnostico.to_string())

Y encima de eso, los hallazgos que una tabla de conteos no puede decir sola y
que son el valor de lo que hiciste hoy:

| Columna | Hallazgo | Cómo se encontró |
|---|---|---|
| `id` | 1 duplicado (`1021`, fila completa repetida) | `duplicated` |
| `producto` | 8 valores distintos para 5 productos; las laptops son 9, no 6 | `value_counts` + `.str.strip().str.capitalize()` |
| `monto_venta` | cargó como texto por `"no disponible"`; 1 vacío, 1 no numérico; mínimo -250 y máximo 999,999 | `dtypes` + `to_numeric` |
| `minutos_en_sitio` | 2 vacíos y un **-3** imposible | `info` + `describe` |
| `timestamp` | 6 formatos distintos; 18 en ISO, 5 en otros 5 formatos, 1 vacío | `.str.extract` + `.str.contains` |
| `email` | 18 válidos, 5 mal formados, 1 vacío; 1 repetido | `.str.fullmatch` |
| `telefono` | 22 recuperables de 9 formatos, 1 incompleto, 1 vacío | `.str.replace` + `.str.len` |

> **Así se ve un diagnóstico y así se entrega.** Números que suman, cada uno con
> la llamada que lo produjo, y la distinción entre *falta*, *está mal escrito* y
> *está mal escrito pero se puede recuperar*. Nada de esto corrigió un solo
> valor del dataset, y es a propósito: el reporte de la P4 es la medición. La
> corrección viene después, y se mide contra esto.

### Ejercicios sesión 3

Del 11 al 15 se corren sobre **tu** dataset completo. El 16 es el integrador: lo
que salga de ahí es el borrador de tu entrega de la P4.

In [ ]:
# Ejercicio 11: Sobre tu columna email, calcula los tres numeros que suman el
# total: vacios, validos y mal formados, con el RE_EMAIL que escribiste en la
# semana 8. Verifica que sumen tu numero de filas. Luego imprime 10 de los mal
# formados y clasificalos a mano por tipo de error (sin arroba, dos arrobas,
# sin dominio, sin extension, con espacios).

# Tu codigo aqui:

In [ ]:
# Ejercicio 12: Sobre tu columna telefono, haz el conteo de longitudes con
# .str.replace(r"\D", "", regex=True).str.len().value_counts().sort_index() y
# escribe, en un comentario, tu criterio: que largos aceptas como validos, cual
# recuperas y cual declaras irrecuperable. Luego clasifica la columna con
# np.select y reporta el conteo de cada estado.

# Tu codigo aqui:

In [ ]:
# Ejercicio 13: Sobre tu columna timestamp, arma tu propio diccionario FORMATOS
# con los formatos que de verdad aparecen (usa .str.extract del ISO para aislar
# los que no cumplen y mirarlos). Cuenta cuantos hay de cada uno y VERIFICA que
# la suma mas los vacios de tu numero de filas. Si te pasa del total, tienes
# patrones que se pisan: ancla con ^.

# Tu codigo aqui:

In [ ]:
# Ejercicio 14: Sobre tu columna <categorica>, compara value_counts() antes y
# despues de .str.strip().str.lower(). Reporta cuantos valores distintos tenias,
# cuantos quedan, y que categoria cambio mas de cuenta. No modifiques la columna.

# Tu codigo aqui:

In [ ]:
# Ejercicio 15: Cuenta los duplicados de tu dataset de dos formas, porque no son
# lo mismo:
#   - filas identicas en TODAS las columnas: df.duplicated().sum()
#   - ids repetidos: df["id"].duplicated().sum()
# Si los dos numeros no coinciden, hay filas con el mismo id y datos distintos.
# Imprime un par con duplicated(keep=False) y di en que columnas difieren.

# Tu codigo aqui:

In [ ]:
# Ejercicio 16 (integrador): Escribe un script que lea tu datos/{tema}-ruido.csv
# y produzca la tabla de diagnostico de tu dataset: una fila por columna con
# dtype, no_vacios, vacios, pct_vacios y distintos, mas los hallazgos
# especificos de tus columnas email, telefono, timestamp y <categorica>.
#
# Guardalo en un .py y que imprima el reporte. Verifica tres cosas antes de
# darlo por bueno:
#   - que los tres numeros de email sumen tu total de filas
#   - que la suma de tus FORMATOS de timestamp mas los vacios de tu total
#   - que ninguna columna numerica haya quedado como texto sin que lo reportes
#
# Esto es el borrador del entregable de la P4. Guardalo.

# Tu codigo aqui:

---

## Chuleta de la semana

**Cargar y mirar**

| Llamada | Da |
|---|---|
| `pd.read_csv(ruta, sep="\|")` | el `DataFrame` |
| `df.shape` · `len(df)` | (filas, columnas) · filas |
| `df.columns.tolist()` | los nombres |
| `df.dtypes` | el tipo de cada columna |
| `df.info()` | tipos **y** `Non-Null Count` de un golpe |
| `df.head(n)` · `df.tail(n)` | las primeras · las últimas |
| `df.describe()` | estadística de las numéricas |
| `df[["a","b"]].describe()` | `count`/`unique`/`top`/`freq` de las de texto |

**Seleccionar**

| Escribo | Obtengo |
|---|---|
| `df["col"]` | una `Series` |
| `df[["col"]]` · `df[["a","b"]]` | un `DataFrame` |
| `df.loc[etiqueta, "col"]` | por **etiqueta** — el final del rango **sí** entra |
| `df.iloc[posicion, n_columna]` | por **posición** — el final **no** entra |
| `df.loc[mascara, ["a","b"]]` | filas por condición y columnas escogidas |
| `df.set_index("id")` | la columna como índice |

**Filtrar**

| Quiero | Escribo |
|---|---|
| una condición | `df["col"] > 10` |
| dos condiciones | `(cond1) & (cond2)` — nunca `and` |
| alguna | `(cond1) \| (cond2)` |
| lo contrario | `~cond` |
| pertenece a una lista | `df["col"].isin([...])` |
| en un rango | `df["col"].between(a, b)` |
| falta / no falta | `df["col"].isna()` · `.notna()` |
| texto a número | `pd.to_numeric(df["col"], errors="coerce")` |

**Contar y ordenar**

| Llamada | Da |
|---|---|
| `df["col"].value_counts(dropna=False)` | cuántos de cada valor, huecos incluidos |
| `.value_counts(normalize=True)` | en proporción |
| `df["col"].nunique()` · `.unique()` | cuántos distintos · cuáles |
| `df.sort_values("col", ascending=False)` | ordenada, tabla nueva |
| `df.nlargest(n, "col")` · `nsmallest` | los n más altos · más bajos |
| `df.duplicated(keep=False)` | marca **todas** las apariciones repetidas |

**El accessor `.str`** (la semana 8, por columna)

| Llamada | Da |
|---|---|
| `.str.len()` · `.str.strip()` · `.str.lower()` | lo de siempre, vectorizado |
| `.str.fullmatch(p)` | máscara: toda la celda cumple |
| `.str.contains(p)` | máscara: el patrón aparece adentro |
| `.str.extract(p)` | **`DataFrame`** con una columna por grupo |
| `.str.replace(p, x, regex=True)` | sustituye — **`regex=True` o no hace nada** |
| `.str.startswith(s)` | máscara |

**Asignar**

```python
df["nueva"] = df["a"] / df["b"]            # columna nueva, vectorizada
df.loc[mascara, "col"] = valor             # modificar las filas que cumplen
sub = df[mascara].copy()                   # un subconjunto aparte
df[mascara]["col"] = valor                 # NO HACE NADA
```

---

## Resumen de la semana

**La forma de trabajar**, que es lo que hay que llevarse:

1. Carga con el `sep` correcto y **mira** antes de calcular: `shape`, `dtypes`,
   `info`, `describe`.
2. Una columna numérica que cargó como texto no es un detalle: es el aviso de
   que hay basura adentro, y `to_numeric(errors="coerce")` te dice en qué filas.
3. Antes de contar por categoría, `value_counts()`. Si hay más valores
   distintos que categorías de verdad, cualquier conteo que hagas está mal.
4. Para validar texto, el patrón de la semana 8 con `.str.fullmatch`. Y después
   **mira los rechazados**: te dicen si el dataset está roto o si el patrón lo
   está.
5. Separa siempre tres cosas: lo que **falta**, lo que está **mal formado** y lo
   que está mal escrito pero es **recuperable**. Los tres números suman el total,
   y esa suma es tu prueba de que no se te perdió nada.
6. Esta semana **mide**; no corrige. Lo que no mediste no lo puedes arreglar, y
   lo que arreglaste sin medir no lo puedes defender.

**Los tres idiomas que más vas a usar**

```python
df.loc[mascara, ["id", "col"]]                      # filtrar y escoger columnas
df["col"].value_counts(dropna=False)                # el conteo con los huecos
df["col"].str.fullmatch(PATRON) & df["col"].notna() # valido Y presente
```

## Las cinco trampas de esta semana

1. **Olvidar `sep="|"`.** No da error: te deja un `DataFrame` de una sola
   columna con toda la cabecera como nombre. Si `df.shape` dice `(103000, 1)`,
   es esto.
2. **`and` / `or` entre máscaras, o condiciones sin paréntesis.** La primera da
   `ValueError: The truth value of a Series is ambiguous`; la segunda, un
   `TypeError` que no se parece a nada. Son `&`, `|`, `~`, y cada condición en su
   paréntesis.
3. **Asignar sobre un filtro en una línea.** `df[mascara]["col"] = valor` no hace
   **nada** —modifica una tabla temporal que se descarta. Es
   `df.loc[mascara, "col"] = valor`.
4. **`.str.replace` sin `regex=True`.** Busca el patrón literalmente, no
   encuentra nada, y te devuelve la columna idéntica sin avisar. Es la `r"..."`
   olvidada de la semana 8, en versión pandas.
5. **Reportar los vacíos como mal formados.** `.str.fullmatch` da `False` tanto
   para `"luis@@gmail.com"` como para la celda vacía. Son dos problemas
   distintos; cuéntalos aparte y verifica que sumen el total.

## Qué sigue

**La semana 10: pandas II, limpieza y transformación.** Todo lo que esta semana
quedó "medido pero sin tocar" es el programa de la que entra:

| Lo que hoy contaste | La semana que entra |
|---|---|
| `isna().sum()` por columna | `fillna`, `dropna` y la decisión de cuál va dónde |
| `to_numeric(errors="coerce")` en una variable | `astype` y la columna convertida de verdad |
| `.str.strip().str.capitalize()` en una variable | la columna normalizada, y qué se rompe al hacerlo |
| `duplicated().sum()` | `drop_duplicates` y qué fila se queda |
| el catálogo de `FORMATOS` | `pd.to_datetime` y las fechas ambiguas |
| `value_counts()` por categoría | `groupby` + `agg`: la estadística **por** grupo |

**Y de ahí la Práctica 4**, que son las dos mitades juntas: el diagnóstico de
calidad de tu dataset (esta semana) y la validación de `email` y `telefono` con
regex (la semana 8). El enunciado se publica en `labs/P4/`.

Dos cosas que conviene dejar listas desde hoy:

- el **script del ejercicio 16** corriendo sobre tus 103,000 filas, con sus tres
  verificaciones de suma
- los hallazgos de tus columnas escritos en una frase cada uno, como van en el
  README: qué encontraste, con qué llamada, y cuántos son